<a href="https://colab.research.google.com/github/Jaydeep-Nadkarni/Custom-LLM/blob/main/LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!nvidia-smi


Sun Oct  4 05:26:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    print("GPU not detected. Check your Colab runtime settings.")


PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [3]:

!git clone https://github.com/Jaydeep-Nadkarni/Custom-LLM.git


Cloning into 'Custom-LLM'...
remote: Enumerating objects: 172, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 172 (delta 73), reused 140 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (172/172), 640.48 KiB | 18.84 MiB/s, done.
Resolving deltas: 100% (73/73), done.


In [4]:

%cd /content/Custom-LLM
!ls -la



/content/Custom-LLM
total 96
drwxr-xr-x 12 root root 4096 Oct  4 05:27 .
drwxr-xr-x  1 root root 4096 Oct  4 05:27 ..
drwxr-xr-x  2 root root 4096 Oct  4 05:27 checkpoints
drwxr-xr-x  2 root root 4096 Oct  4 05:27 configs
drwxr-xr-x  6 root root 4096 Oct  4 05:27 data
drwxr-xr-x  2 root root 4096 Oct  4 05:27 evaluation
drwxr-xr-x  8 root root 4096 Oct  4 05:27 .git
-rw-r--r--  1 root root  215 Oct  4 05:27 .gitignore
drwxr-xr-x  2 root root 4096 Oct  4 05:27 inference
drwxr-xr-x  2 root root 4096 Oct  4 05:27 logs
drwxr-xr-x  2 root root 4096 Oct  4 05:27 model
-rw-r--r--  1 root root  857 Oct  4 05:27 README.md
-rw-r--r--  1 root root   78 Oct  4 05:27 requirements.txt
-rw-r--r--  1 root root  563 Oct  4 05:27 test_attention.py
-rw-r--r--  1 root root 1306 Oct  4 05:27 test_backward.py
-rw-r--r--  1 root root  622 Oct  4 05:27 test_causal_attention.py
-rw-r--r--  1 root root  527 Oct  4 05:27 test_dataloader.py
-rw-r--r--  1 root root  660 Oct  4 05:27 test_embedding.py
-rw-r--r--  1

In [5]:

!ls requirements.txt


requirements.txt


In [6]:

%pip install -r requirements.txt


In [7]:

!find data -maxdepth 3 -type f | head -50


data/scripts/filter.py
data/scripts/deduplicate.py
data/scripts/download.py
data/scripts/clean.py
data/scripts/tokenize.py
data/tokenized/.gitkeep
data/raw/.gitkeep
data/cleaned/.gitkeep
data/dataset.py


In [8]:

!ls model training data
!ls configs


data:
cleaned  dataset.py  raw  scripts  tokenized

model:
attention.py   __init__.py  rmsnorm.py	swiglu.py
embeddings.py  model.py     rope.py	transformer_block.py

training:
checkpoint.py  config.py  __init__.py  optimizer.py  scheduler.py  train.py
100m.yaml  50m.yaml  debug.yaml


In [9]:
!find /content -type f \( \
  -name "fineweb_train_inputs.npy" -o \
  -name "fineweb_train_targets.npy" -o \
  -name "fineweb_train_metadata.json" \
\) 2>/dev/null

/content/Custom-LLM/data/tokenized/fineweb_train_targets.npy
/content/Custom-LLM/data/tokenized/fineweb_train_metadata.json
/content/Custom-LLM/data/tokenized/fineweb_train_inputs.npy


In [10]:

import os
import json
import numpy as np
import torch

DATA_DIR = "/content/Custom-LLM/data/tokenized"

inputs_path = os.path.join(DATA_DIR, "fineweb_train_inputs.npy")
targets_path = os.path.join(DATA_DIR, "fineweb_train_targets.npy")
metadata_path = os.path.join(DATA_DIR, "fineweb_train_metadata.json")

for path in [inputs_path, targets_path, metadata_path]:
    assert os.path.isfile(path), f"Missing file: {path}"
    print(f"Found: {os.path.basename(path)}")

inputs = np.load(inputs_path, mmap_mode="r")
targets = np.load(targets_path, mmap_mode="r")

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("\nInput shape:", inputs.shape)
print("Target shape:", targets.shape)
print("Input dtype:", inputs.dtype)
print("Target dtype:", targets.dtype)
print("Metadata:", metadata)

assert inputs.shape == targets.shape, "Input/target shapes do not match!"
assert inputs.ndim == 2 and inputs.shape[1] == 512, "Unexpected sequence shape!"

print("\nCUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    print("STOP: Enable a GPU runtime before training.")


Found: fineweb_train_inputs.npy
Found: fineweb_train_targets.npy
Found: fineweb_train_metadata.json

Input shape: (19625, 512)
Target shape: (19625, 512)
Input dtype: uint16
Target dtype: uint16
Metadata: {'source': 'D:\\Programming\\AIML\\LLM\\data\\cleaned\\fineweb_deduplicated.jsonl', 'tokenizer': 'D:\\Programming\\AIML\\LLM\\tokenizer\\tokenizer.json', 'vocab_size': 32000, 'eos_id': 3, 'block_size': 512, 'sequence_count': 19625, 'discarded_tokens': 56, 'dtype': 'uint16'}

CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [11]:

%cd /content/Custom-LLM

print("===== DEBUG CONFIG =====")
!cat configs/debug.yaml

print("\n===== TRAIN SCRIPT OPTIONS =====")
!python training/train.py --help


/content/Custom-LLM
===== DEBUG CONFIG =====
model:
  vocab_size: 32000
  d_model: 384
  n_heads: 6
  n_layers: 6
  hidden_dim: 1024
  context_length: 512
  dropout: 0.0
training:
  batch_size: 8
  steps: 10000
  learning_rate: 0.0003
  weight_decay: 0.1
  warmup_steps: 500
  log_every: 100
data:
  train_inputs: data/tokenized/fineweb_train_inputs.npy
  train_targets: data/tokenized/fineweb_train_targets.npy
===== TRAIN SCRIPT OPTIONS =====
usage: train.py [-h] [--config CONFIG] [--steps STEPS]
                [--batch-size BATCH_SIZE] [--context-length CONTEXT_LENGTH]
                [--log-every LOG_EVERY] [--checkpoint-every CHECKPOINT_EVERY]
                [--log-file LOG_FILE]

options:
  -h, --help            show this help message and exit
  --config CONFIG
  --steps STEPS
  --batch-size BATCH_SIZE
  --context-length CONTEXT_LENGTH
  --log-every LOG_EVERY
  --checkpoint-every CHECKPOINT_EVERY
  --log-file LOG_FILE


In [12]:
%cd /content/Custom-LLM

print("===== TRAINING SCRIPT =====")
!sed -n '1,280p' training/train.py

print("\n===== CHECKPOINT IMPLEMENTATION =====")
!cat training/checkpoint.py

/content/Custom-LLM
===== TRAINING SCRIPT =====
import argparse
from pathlib import Path
import sys

import torch
import torch.nn.functional as F
import yaml
from torch.utils.data import DataLoader

PROJECT_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from data.dataset import TokenDataset
from model.model import TransformerLM
from training.optimizer import build_optimizer
from training.scheduler import build_scheduler
from training.checkpoint import save_checkpoint


def main() -> None:
    # -----------------------------
    # Load configuration
    # -----------------------------

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--config",
        type=Path,
        default=Path("configs/debug.yaml")
    )
    parser.add_argument("--steps", type=int, default=None)
    parser.add_argument("--batch-size", type=int, default=None)
    parser.add_argument("--context-length", type=int, default=None)
    parser.add_argument("--lo

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
%cd /content/Custom-LLM

from pathlib import Path

path = Path("training/train.py")
source = path.read_text()

source = source.replace(
    'parser.add_argument("--log-file", type=Path, default=Path("logs/training.log"))',
    'parser.add_argument("--log-file", type=Path, default=Path("logs/training.log"))\n'
    '    parser.add_argument("--checkpoint-path", type=Path, default=Path("checkpoints/latest.pt"))'
)

source = source.replace(
    'Path("checkpoints/latest.pt"),',
    'args.checkpoint_path,'
)

path.write_text(source)

print("Updated training script.")
!python training/train.py --help

/content/Custom-LLM
Updated training script.
usage: train.py [-h] [--config CONFIG] [--steps STEPS]
                [--batch-size BATCH_SIZE] [--context-length CONTEXT_LENGTH]
                [--log-every LOG_EVERY] [--checkpoint-every CHECKPOINT_EVERY]
                [--log-file LOG_FILE] [--checkpoint-path CHECKPOINT_PATH]

options:
  -h, --help            show this help message and exit
  --config CONFIG
  --steps STEPS
  --batch-size BATCH_SIZE
  --context-length CONTEXT_LENGTH
  --log-every LOG_EVERY
  --checkpoint-every CHECKPOINT_EVERY
  --log-file LOG_FILE
  --checkpoint-path CHECKPOINT_PATH


In [15]:
%cd /content/Custom-LLM

!python training/train.py \
  --config configs/debug.yaml \
  --steps 100 \
  --log-every 10 \
  --checkpoint-every 50 \
  --checkpoint-path /content/drive/MyDrive/Custom-LLM/checkpoints/test_latest.pt \
  --log-file /content/drive/MyDrive/Custom-LLM/logs/test_training.log

/content/Custom-LLM
Device: cuda
Model parameters: 22909824
Dataset size: 19625
step=1 loss=10.4796 lr=0.00000060 grad_norm=2.6350
step=10 loss=10.4441 lr=0.00000600 grad_norm=2.8309
step=20 loss=10.3376 lr=0.00001200 grad_norm=2.8883
step=30 loss=10.1746 lr=0.00001800 grad_norm=2.7734
step=40 loss=10.0133 lr=0.00002400 grad_norm=2.4801
step=50 loss=9.8130 lr=0.00003000 grad_norm=2.0383
step=60 loss=9.6792 lr=0.00003600 grad_norm=1.7265
step=70 loss=9.4993 lr=0.00004200 grad_norm=1.7476
step=80 loss=9.3343 lr=0.00004800 grad_norm=1.7954
step=90 loss=9.1893 lr=0.00005400 grad_norm=1.7159
step=100 loss=8.9858 lr=0.00006000 grad_norm=1.7319
Training smoke run completed: 100 steps


In [16]:
%cd /content/Custom-LLM

from pathlib import Path

path = Path("training/train.py")
source = path.read_text()

source = source.replace(
    "from training.checkpoint import save_checkpoint",
    "from training.checkpoint import save_checkpoint, load_checkpoint"
)

source = source.replace(
    'parser.add_argument("--checkpoint-path", type=Path, default=Path("checkpoints/latest.pt"))',
    'parser.add_argument("--checkpoint-path", type=Path, default=Path("checkpoints/latest.pt"))\n'
    '    parser.add_argument("--resume-from", type=Path, default=None)'
)

source = source.replace(
    '    args.log_file.parent.mkdir(parents=True, exist_ok=True)\n    model.train()\n    data_iterator = iter(dataloader)\n    with args.log_file.open("a", encoding="utf-8") as log_file:\n        for step in range(1, total_steps + 1):',
    '''    start_step = 0
    if args.resume_from is not None:
        start_step = load_checkpoint(
            args.resume_from, model, optimizer, scheduler
        )
        print(f"Resuming from step {start_step}")

    args.log_file.parent.mkdir(parents=True, exist_ok=True)
    model.train()
    data_iterator = iter(dataloader)
    with args.log_file.open("a", encoding="utf-8") as log_file:
        for step in range(start_step + 1, total_steps + 1):'''
)

path.write_text(source)

print("Resume support added.")
!python training/train.py --help

/content/Custom-LLM
Resume support added.
usage: train.py [-h] [--config CONFIG] [--steps STEPS]
                [--batch-size BATCH_SIZE] [--context-length CONTEXT_LENGTH]
                [--log-every LOG_EVERY] [--checkpoint-every CHECKPOINT_EVERY]
                [--log-file LOG_FILE] [--checkpoint-path CHECKPOINT_PATH]
                [--resume-from RESUME_FROM]

options:
  -h, --help            show this help message and exit
  --config CONFIG
  --steps STEPS
  --batch-size BATCH_SIZE
  --context-length CONTEXT_LENGTH
  --log-every LOG_EVERY
  --checkpoint-every CHECKPOINT_EVERY
  --log-file LOG_FILE
  --checkpoint-path CHECKPOINT_PATH
  --resume-from RESUME_FROM


In [17]:
%cd /content/Custom-LLM

!python training/train.py \
  --config configs/debug.yaml \
  --steps 10000 \
  --batch-size 8 \
  --log-every 100 \
  --checkpoint-every 500 \
  --checkpoint-path /content/drive/MyDrive/Custom-LLM/checkpoints/latest.pt \
  --log-file /content/drive/MyDrive/Custom-LLM/logs/training.log

/content/Custom-LLM
Device: cuda
Model parameters: 22909824
Dataset size: 19625
step=1 loss=10.4437 lr=0.00000060 grad_norm=2.6358
step=100 loss=8.9843 lr=0.00006000 grad_norm=1.7521
step=200 loss=7.3158 lr=0.00012000 grad_norm=0.6449
step=300 loss=6.9855 lr=0.00018000 grad_norm=0.7314
step=400 loss=6.8958 lr=0.00024000 grad_norm=0.8353
step=500 loss=7.0778 lr=0.00030000 grad_norm=0.8477
step=600 loss=6.7439 lr=0.00029842 grad_norm=0.7570
step=700 loss=6.4258 lr=0.00029684 grad_norm=0.8875
step=800 loss=6.3555 lr=0.00029526 grad_norm=1.0414
step=900 loss=5.9413 lr=0.00029368 grad_norm=0.9536
step=1000 loss=6.3899 lr=0.00029211 grad_norm=0.9556
step=1100 loss=6.1164 lr=0.00029053 grad_norm=0.9200
step=1200 loss=6.0535 lr=0.00028895 grad_norm=0.9945
step=1300 loss=5.8383 lr=0.00028737 grad_norm=0.8667
step=1400 loss=6.0966 lr=0.00028579 grad_norm=1.0819
step=1500 loss=5.8407 lr=0.00028421 grad_norm=0.9618
step=1600 loss=5.8504 lr=0.00028263 grad_norm=0.8965
step=1700 loss=5.6304 lr=0.000

In [18]:
%cd /content/Custom-LLM

from pathlib import Path

checkpoint = Path(
    "/content/drive/MyDrive/Custom-LLM/checkpoints/latest.pt"
)

print("Checkpoint exists:", checkpoint.exists())

if checkpoint.exists():
    print("Checkpoint size (MB):", round(checkpoint.stat().st_size / 1e6, 2))

print("\n===== GENERATION SCRIPT =====")
!sed -n '1,260p' inference/generate.py

print("\n===== EVALUATION SCRIPT =====")
!sed -n '1,260p' evaluation/evaluate.py

print("\n===== PERPLEXITY SCRIPT =====")
!sed -n '1,220p' evaluation/perplexity.py

/content/Custom-LLM
Checkpoint exists: True
Checkpoint size (MB): 275.76

===== GENERATION SCRIPT =====
import argparse
from pathlib import Path
import torch
from tokenizers import Tokenizer
from model import DecoderLM
from inference.sampling import sample_next_token


def generate(model, prompt_ids: list[int], max_new_tokens: int, temperature: float, top_k: int | None) -> list[int]:
    tokens = torch.tensor([prompt_ids], dtype=torch.long)
    for _ in range(max_new_tokens):
        logits, _ = model(tokens)
        next_token = sample_next_token(logits[:, -1, :], temperature, top_k)
        tokens = torch.cat((tokens, next_token), dim=1)
    return tokens[0].tolist()


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", type=Path, default=Path("checkpoints/latest.pt"))
    parser.add_argument("--prompt", default="Hello")
    parser.add_argument("--max-new-tokens", type=int, default=50)
    parser.add_argument("--temperature", type=float,

In [19]:
%cd /content/Custom-LLM

print("===== MODEL DEFINITION =====")
!sed -n '1,280p' model/model.py

print("\n===== TOKENIZER SPECIAL TOKENS =====")
!cat tokenizer/tokenizer.json | head -c 1200

print("\n\n===== SAMPLING IMPLEMENTATION =====")
!cat inference/sampling.py

/content/Custom-LLM
===== MODEL DEFINITION =====
import torch
import torch.nn as nn
import torch.nn.functional as F
from .embeddings import TokenEmbedding
from .rmsnorm import RMSNorm
from .transformer_block import TransformerBlock


class TransformerLM(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        n_layers: int,
        n_heads: int,
        ffn_hidden_size: int,
        max_seq_len: int = 512,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        self.embedding = TokenEmbedding(
            vocab_size=vocab_size,
            hidden_size=d_model,
        )
        self.layers = nn.ModuleList([
            TransformerBlock(
                d_model=d_model,
                n_heads=n_heads,
                ffn_hidden_size=ffn_hidden_size,
                dropout=dropout,
            )
            for _ in range(n

In [20]:
%cd /content/Custom-LLM

print("===== MODEL DEFINITION =====")
!cat model/model.py

print("\n===== TRANSFORMER BLOCK =====")
!cat model/transformer_block.py

print("\n===== CHECKPOINT CONFIG =====")
import torch

path = "/content/drive/MyDrive/Custom-LLM/checkpoints/latest.pt"
state = torch.load(path, map_location="cpu", weights_only=False)

print("Checkpoint step:", state["step"])
print("Config:", state["config"])
print("Model state keys (first 15):")
print(list(state["model"].keys())[:15])

/content/Custom-LLM
===== MODEL DEFINITION =====
import torch
import torch.nn as nn
import torch.nn.functional as F
from .embeddings import TokenEmbedding
from .rmsnorm import RMSNorm
from .transformer_block import TransformerBlock


class TransformerLM(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        n_layers: int,
        n_heads: int,
        ffn_hidden_size: int,
        max_seq_len: int = 512,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        self.embedding = TokenEmbedding(
            vocab_size=vocab_size,
            hidden_size=d_model,
        )
        self.layers = nn.ModuleList([
            TransformerBlock(
                d_model=d_model,
                n_heads=n_heads,
                ffn_hidden_size=ffn_hidden_size,
                dropout=dropout,
            )
            for _ in range(n

In [21]:
%cd /content/Custom-LLM

from pathlib import Path

code = r'''
import argparse
from pathlib import Path

import torch
from tokenizers import Tokenizer

from model.model import TransformerLM
from inference.sampling import sample_next_token


@torch.no_grad()
def generate(model, prompt_ids, max_new_tokens, temperature, top_k, device):
    model.eval()
    tokens = torch.tensor([prompt_ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        context = tokens[:, -model.max_seq_len:]
        logits = model(context)
        next_token = sample_next_token(
            logits[:, -1, :],
            temperature=temperature,
            top_k=top_k,
        )
        tokens = torch.cat((tokens, next_token), dim=1)

        if next_token.item() == 3:  # <EOS>
            break

    return tokens[0].tolist()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", type=Path,
                        default=Path("checkpoints/latest.pt"))
    parser.add_argument("--prompt", default="Hello")
    parser.add_argument("--max-new-tokens", type=int, default=50)
    parser.add_argument("--temperature", type=float, default=0.8)
    parser.add_argument("--top-k", type=int, default=50)
    args = parser.parse_args()

    if args.max_new_tokens < 1:
        parser.error("--max-new-tokens must be positive")
    if args.temperature <= 0:
        parser.error("--temperature must be positive")
    if args.top_k < 1:
        parser.error("--top-k must be positive")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    state = torch.load(
        args.checkpoint, map_location="cpu", weights_only=False
    )
    cfg = state["config"]["model"]

    model = TransformerLM(
        vocab_size=cfg["vocab_size"],
        d_model=cfg["d_model"],
        n_layers=cfg["n_layers"],
        n_heads=cfg["n_heads"],
        ffn_hidden_size=cfg["hidden_dim"],
        max_seq_len=cfg["context_length"],
        dropout=cfg["dropout"],
    )
    model.load_state_dict(state["model"])
    model.to(device)
    model.eval()

    tokenizer = Tokenizer.from_file("tokenizer/tokenizer.json")
    prompt_ids = tokenizer.encode(args.prompt).ids
    if not prompt_ids:
        raise ValueError("Prompt produced no tokens")

    output_ids = generate(
        model, prompt_ids, args.max_new_tokens,
        args.temperature, args.top_k, device
    )
    print("\n--- Generated text ---")
    print(tokenizer.decode(output_ids))
    print("----------------------")


if __name__ == "__main__":
    main()
'''

Path("inference/generate.py").write_text(code, encoding="utf-8")
print("Generation script updated.")

/content/Custom-LLM
Generation script updated.


In [27]:
!python inference/generate.py \
  --checkpoint /content/drive/MyDrive/Custom-LLM/checkpoints/latest.pt \
  --prompt "Artificial intelligence is" \
  --max-new-tokens 80 \
  --temperature 0.8 \
  --top-k 40

Device: cuda

--- Generated text ---
Artificial intelligence is an alternative technology that can be used as a method of computer models, which can enable us to create robust, reliable, and convenient algorithms. This technology is not only about tools, but also about the different types of algorithms and the types of algorithms that we can use to make data-driven decisions. However, most importantly, this technology has become a common and valuable solution for the future of AI.
----------------------


In [36]:
!python inference/generate.py \
  --checkpoint /content/drive/MyDrive/Custom-LLM/checkpoints/latest.pt \
  --prompt "The cat is an " \
  --max-new-tokens 30 \
  --temperature 0.5 \
  --top-k 20

Device: cuda

--- Generated text ---
The cat is an ure or just, but it is not a little bit. It’s a great thing – it’s a great thing! So, don’
----------------------
